In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF

In [ ]:
# Original dataset has 40 rows

In [5]:
X = np.load(r"initial_data\function_8\initial_inputs.npy")
Y = np.load(r"initial_data\function_8\initial_outputs.npy")
df = pd.DataFrame({
    'x1': X[:,0],
    'x2': X[:,1],
    'x3': X[:,2],
    'x4': X[:,3],
    'x5': X[:,4],
    'x6': X[:,5],
    'x7': X[:,6],
    'x8': X[:,7],
    'y': Y
})
df.index = df.index+1
df

,x1,x2,x3,x4,x5,x6,x7,x8,y
1,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
2,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
3,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
4,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
5,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
6,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
7,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
8,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
9,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
10,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [6]:
print(f'Best output value so far:\n {max(Y)}')
print(f'Corresponding x values:\n {X[np.argmax(Y)]}')
print(f'Best query output:\n Query {np.argmax(Y[40:])+1}: {max(Y[40:])}')
print(f'Improvement over last:\n {max(Y)} - {max(Y[:(len(Y)-1)])} = {max(Y)-max(Y[:(len(Y)-1)])}')

Best output value so far:
 9.9671646985596
Corresponding x values:
 [0.119143 0.135448 0.151215 0.240738 0.752919 0.368148 0.224161 0.437327]
Best query output:
 Query 6: 9.9671646985596
Improvement over last:
 9.9671646985596 - 9.9671646985596 = 0.0


In [ ]:
kernel = ConstantKernel(1.0) * RBF(length_scale=0.2)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state = 42
)

gp.fit(X, Y)

candidate_points = np.random.rand(10000,8)

mu, sigma = gp.predict(candidate_points, return_std=True)

best_y = np.max(Y)
xi = 0

improvement = mu - best_y - xi

Z = np.zeros_like(mu)
mask = sigma > 0
Z[mask] = improvement[mask] / sigma[mask]

EI = np.zeros_like(mu)
EI[mask] = (
    improvement[mask] * norm.cdf(Z[mask])
    + sigma[mask] * norm.pdf(Z[mask])
)

next_point = candidate_points[np.argmax(EI)]

print("Next point:", np.round(next_point, 6))

mean, std = gp.predict(next_point.reshape(-1,8), return_std=True)
print(f'Predicted mean: {mean}')
print(f'Predicted std: {std}')

Next point: [0.100726 0.348892 0.286628 0.155801 0.996472 0.488789 0.042264 0.21525 ]
Predicted mean: [9.82388286]
Predicted std: [0.07782166]


In [6]:
def plot_EI_slice(gp, X, Y, dim1=0, dim2=1, resolution=100, xi=0):
    """
    Plot a 2D slice of the Expected Improvement acquisition function.

    Parameters
    ----------
    gp : fitted GaussianProcessRegressor
    X : observed inputs (n_samples, n_features)
    Y : observed outputs
    dim1 : first variable to plot
    dim2 : second variable to plot
    """

    best_point = X[np.argmax(Y)].copy()

    x = np.linspace(0, 1, resolution)
    y = np.linspace(0, 1, resolution)

    xx, yy = np.meshgrid(x, y)

    grid = np.tile(best_point, (xx.size, 1))

    grid[:, dim1] = xx.ravel()
    grid[:, dim2] = yy.ravel()

    mu, sigma = gp.predict(grid, return_std=True)

    best_y = np.max(Y)

    improvement = mu - best_y - xi

    EI = np.zeros_like(mu)

    mask = sigma > 0

    Z = np.zeros_like(mu)
    Z[mask] = improvement[mask] / sigma[mask]

    EI[mask] = (
        improvement[mask] * norm.cdf(Z[mask])
        + sigma[mask] * norm.pdf(Z[mask])
    )

    plt.figure(figsize=(7,6))

    plt.contourf(xx, yy, EI.reshape(xx.shape), levels=30, cmap="viridis")

    plt.scatter(
        X[:, dim1],
        X[:, dim2],
        c="red",
        edgecolor="black"
    )

    plt.scatter(
    best_point[dim1],
    best_point[dim2],
    c="yellow",
    edgecolor="black",
    s=200,
    marker="*",
    linewidth=1.5,
    label="Current best"
    )

    plt.xlabel(f"x{dim1+1}")
    plt.ylabel(f"x{dim2+1}")

    plt.colorbar(label="Expected Improvement")

    plt.show()

plot_EI_slice(gp, X, Y, dim1=4, dim2=0)
plot_EI_slice(gp, X, Y, dim1=4, dim2=1)
plot_EI_slice(gp, X, Y, dim1=4, dim2=2)
plot_EI_slice(gp, X, Y, dim1=4, dim2=3)
plot_EI_slice(gp, X, Y, dim1=4, dim2=5)
plot_EI_slice(gp, X, Y, dim1=4, dim2=6)
plot_EI_slice(gp, X, Y, dim1=4, dim2=7)

NameError: name 'gp' is not defined

In [ ]:
'''
1st point selected: [0.107783-0.152192-0.201339-0.074624-0.811699-0.315491-0.261762-0.653263]
Output: 9.9368999564326

2nd point selected: [0.299447-0.364298-0.050798-0.141816-0.849232-0.270688-0.114265-0.117678]
Output: 9.7638719874356

3rd point selected: [0.061165-0.395674-0.011848-0.130983-0.841616-0.924798-0.277544-0.950803]
Output: 9.6887349347881

4th point selected: [0.169275-0.550904-0.032293-0.008837-0.354759-0.280058-0.059641-0.837916]
Output: 9.5885550079459

5th point selected: [0.105393-0.327722-0.057115-0.843613-0.936749-0.917816-0.188227-0.119404]
Output: 9.2640262454379

6th point selected: [0.119143-0.135448-0.151215-0.240738-0.752919-0.368148-0.224161-0.437327]
Output: 9.9671646985596   ---NEW BEST---

7th point selected: [0.058868-0.247276-0.168965-0.013817-0.764793-0.872956-0.106506-0.257485]
Output: 9.795123220857

8th point selected: [0.056699-0.208258-0.222859-0.107653-0.638599-0.644902-0.262863-0.559876]
Output: 9.9231081641019

9th point selected: [0.084801-0.077582-0.135949-0.109361-0.886361-0.405532-0.293292-0.418016]
Output: 9.9591639876119

10th point selected: [0.111558-0.096707-0.205993-0.091482-0.891478-0.528031-0.049636-0.566983]
Output: 9.9258459905281

11th point selected: [0.232492-0.027028-0.067669-0.171789-0.814573-0.274126-0.178082-0.219854]
Output: 9.8711022602639

12th point selected: [0.083281-0.031856-0.188485-0.144949-0.987742-0.697138-0.155439-0.666266] --- xi reduced from 0.01 to 0.005 ---
Output: 9.9142985430224

Final point selected: [0.166613-0.184499-0.155767-0.259642-0.917296-0.552401-0.273491-0.456513] --- xi reduced to 0 ---
Output: 9.9534363057421

In [ ]:
# Updating data with new input-output pairs

# X_query1 = np.array([0.107783,0.152192,0.201339,0.074624,0.811699,0.315491,0.261762,0.653263], dtype=np.float64)
# Y_query1_output = np.array([9.9368999564326], dtype=np.float64)
# X_updated = np.vstack((X, X_query1))
# Y_updated = np.append(Y, Y_query1_output)

# X_query2 = np.array([0.299447,0.364298,0.050798,0.141816,0.849232,0.270688,0.114265,0.117678], dtype=np.float64)
# Y_query2_output = np.array([9.7638719874356], dtype=np.float64)
# X_updated = np.vstack((X, X_query2))
# Y_updated = np.append(Y, Y_query2_output)

# X_query3 = np.array([0.061165,0.395674,0.011848,0.130983,0.841616,0.924798,0.277544,0.950803], dtype=np.float64)
# Y_query3_output = np.array([9.6887349347881], dtype=np.float64)
# X_updated = np.vstack((X, X_query3))
# Y_updated = np.append(Y, Y_query3_output)

# X_query4 = np.array([0.169275,0.550904,0.032293,0.008837,0.354759,0.280058,0.059641,0.837916], dtype=np.float64)
# Y_query4_output = np.array([9.5885550079459], dtype=np.float64)
# X_updated = np.vstack((X, X_query4))
# Y_updated = np.append(Y, Y_query4_output)

# X_query5 = np.array([0.105393,0.327722,0.057115,0.843613,0.936749,0.917816,0.188227,0.119404], dtype=np.float64)
# Y_query5_output = np.array([9.2640262454379], dtype=np.float64)
# X_updated = np.vstack((X, X_query5))
# Y_updated = np.append(Y, Y_query5_output)

# X_query6 = np.array([0.119143,0.135448,0.151215,0.240738,0.752919,0.368148,0.224161,0.437327], dtype=np.float64)
# Y_query6_output = np.array([9.9671646985596], dtype=np.float64)
# X_updated = np.vstack((X, X_query6))
# Y_updated = np.append(Y, Y_query6_output)

# X_query7 = np.array([0.058868,0.247276,0.168965,0.013817,0.764793,0.872956,0.106506,0.257485], dtype=np.float64)
# Y_query7_output = np.array([9.795123220857], dtype=np.float64)
# X_updated = np.vstack((X, X_query7))
# Y_updated = np.append(Y, Y_query7_output)

# X_query8 = np.array([0.056699,0.208258,0.222859,0.107653,0.638599,0.644902,0.262863,0.559876], dtype=np.float64)
# Y_query8_output = np.array([9.9231081641019], dtype=np.float64)
# X_updated = np.vstack((X, X_query8))
# Y_updated = np.append(Y, Y_query8_output)

# X_query9 = np.array([0.084801,0.077582,0.135949,0.109361,0.886361,0.405532,0.293292,0.418016], dtype=np.float64)
# Y_query9_output = np.array([9.9591639876119], dtype=np.float64)
# X_updated = np.vstack((X, X_query9))
# Y_updated = np.append(Y, Y_query9_output)

# X_query10 = np.array([0.111558,0.096707,0.205993,0.091482,0.891478,0.528031,0.049636,0.566983], dtype=np.float64)
# Y_query10_output = np.array([9.9258459905281], dtype=np.float64)
# X_updated = np.vstack((X, X_query10))
# Y_updated = np.append(Y, Y_query10_output)

# X_query11 = np.array([0.232492,0.027028,0.067669,0.171789,0.814573,0.274126,0.178082,0.219854], dtype=np.float64)
# Y_query11_output = np.array([9.8711022602639], dtype=np.float64)
# X_updated = np.vstack((X, X_query11))
# Y_updated = np.append(Y, Y_query11_output)

# X_query12 = np.array([0.083281,0.031856,0.188485,0.144949,0.987742,0.697138,0.155439,0.666266], dtype=np.float64)
# Y_query12_output = np.array([9.9142985430224], dtype=np.float64)
# X_updated = np.vstack((X, X_query12))
# Y_updated = np.append(Y, Y_query12_output)

# X_query13 = np.array([0.166613,0.184499,0.155767,0.259642,0.917296,0.552401,0.273491,0.456513], dtype=np.float64)
# Y_query13_output = np.array([9.9534363057421], dtype=np.float64)
# X_updated = np.vstack((X, X_query13))
# Y_updated = np.append(Y, Y_query13_output)

In [ ]:
# Save new data to original files (initial_inputs.npy & initial_outputs.npy)

# np.save(r"initial_data\function_8\initial_inputs.npy", X_updated)
# np.save(r"initial_data\function_8\initial_outputs.npy", Y_updated)